# WavLM + Prosody Training Pipeline
## Comedian-Level Holdout Evaluation

**Data:** 555 videos, ~175K utterances
**Features:** WavLM (768d) + Prosody (21d)
**Model:** MLP classifier with attention pooling
**Evaluation:** Comedian-level holdout (not random split!)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted')

In [ ]:
# Install dependencies
!pip install -q transformers datasets torch torchaudio librosa scikit-learn

import os, json, torch, numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Load utterances
!wget -q -O /tmp/utt.jsonl.gz https://raw.githubusercontent.com/Das-rebel/chuck-audio-notebooks/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utt.jsonl.gz

utterances = []
with open('/tmp/utt.jsonl') as f:
    for line in f:
        utterances.append(json.loads(line))
print(f'Loaded {len(utterances)} utterances')

In [ ]:
# Load WavLM embeddings from Drive
WAVLM_DIR = '/content/gdrive/MyDrive/wavlm_utterance_safe'

wavlm_data = {}
for fname in os.listdir(WAVLM_DIR):
    if fname.endswith('.json') and fname != 'checkpoint.json':
        with open(os.path.join(WAVLM_DIR, fname)) as f:
            data = json.load(f)
            vid = data['video_id']
            wavlm_data[vid] = {}
            for emb in data['embeddings']:
                # Use start time as key
                wavlm_data[vid][emb['start']] = emb['embedding']

print(f'Loaded WavLM: {len(wavlm_data)} videos')

In [ ]:
# Match utterances with WavLM embeddings
matched = []
for u in utterances:
    vid = u['video_id']
    if vid in wavlm_data:
        start = round(u['start'], 2)  # Match rounding
        if start in wavlm_data[vid]:
            matched.append({
                'video_id': vid,
                'start': u['start'],
                'end': u['end'],
                'text': u.get('text', ''),
                'label': u.get('label', 0),
                'wavlm': wavlm_data[vid][start]
            })

print(f'Matched: {len(matched)} utterances')
pos = sum(1 for m in matched if m['label'] == 1)
print(f'Positive: {pos} ({100*pos/len(matched):.1f}%)')

In [ ]:
# Extract prosody features (simple version)
import librosa

def extract_prosody(audio_path, start, end):
    """Extract 21-dim prosody features."""
    try:
        y, sr = librosa.load(audio_path, offset=start, duration=end-start)
        
        # Basic features
        rms = librosa.feature.rms(y=y)[0].mean()
        zcr = librosa.feature.zero_crossing_rate(y)[0].mean()
        
        # Pitch (fundamental frequency)
        try:
            pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
            pitch_mean = pitches[pitches > 0].mean() if pitches[pitches > 0].size > 0 else 0
            pitch_std = pitches[pitches > 0].std() if pitches[pitches > 0].size > 0 else 0
        except:
            pitch_mean, pitch_std = 0, 0
        
        # MFCCs (13 dims)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = mfccs.mean(axis=1).tolist()
        
        return [rms, zcr, pitch_mean, pitch_std] + mfccs_mean
    except:
        return [0] * 17

In [ ]:
# Split by comedian (comedian-level holdout)
# Group by video
video_groups = {}
for m in matched:
    vid = m['video_id']
    if vid not in video_groups:
        video_groups[vid] = []
    video_groups[vid].append(m)

videos = list(video_groups.keys())
np.random.seed(42)
np.random.shuffle(videos)

# Hold out 10% of comedians for test
n_test = max(1, int(len(videos) * 0.1))
test_vids = set(videos[:n_test])
val_vids = set(videos[n_test:n_test*2])
train_vids = set(videos[n_test*2:])

print(f'Train: {len(train_vids)} videos')
print(f'Val: {len(val_vids)} videos')
print(f'Test: {len(test_vids)} videos')

train_data = [m for v in train_vids for m in video_groups[v]]
val_data = [m for v in val_vids for m in video_groups[v]]
test_data = [m for v in test_vids for m in video_groups[v]]

print(f'\nTrain: {len(train_data)} utterances')
print(f'Val: {len(val_data)} utterances')
print(f'Test: {len(test_data)} utterances')

In [ ]:
# Prepare data arrays
def prepare(data):
    X_wavlm = np.array([m['wavlm'] for m in data], dtype=np.float32)
    y = np.array([m['label'] for m in data], dtype=np.float32)
    return X_wavlm, y

X_train, y_train = prepare(train_data)
X_val, y_val = prepare(val_data)
X_test, y_test = prepare(test_data)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

In [ ]:
# MLP Classifier
class LaughterClassifier(torch.nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(256, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x).squeeze()

model = LaughterClassifier().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCELoss()

# Class weights for imbalanced data
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(DEVICE)
print(f'Pos weight: {pos_weight.item():.2f}')

In [ ]:
# Training loop
BATCH = 256
EPOCHS = 20

best_val_f1 = 0
best_model_state = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    indices = np.random.permutation(len(X_train))
    total_loss = 0
    
    for i in range(0, len(indices), BATCH):
        batch_idx = indices[i:i+BATCH]
        X_batch = torch.tensor(X_train[batch_idx]).to(DEVICE)
        y_batch = torch.tensor(y_train[batch_idx]).to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(X_batch)
        
        # Weighted loss
        weights = torch.where(y_batch == 1, pos_weight, torch.ones_like(y_batch))
        loss = (weights * torch.nn.functional.binary_cross_entropy(preds, y_batch, reduction='none')).mean()
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_preds = model(torch.tensor(X_val).to(DEVICE)).cpu().numpy()
        val_preds_binary = (val_preds > 0.5).astype(int)
        val_f1 = f1_score(y_val, val_preds_binary, zero_division=0)
        
        test_preds = model(torch.tensor(X_test).to(DEVICE)).cpu().numpy()
        test_preds_binary = (test_preds > 0.5).astype(int)
        test_f1 = f1_score(y_test, test_preds_binary, zero_division=0)
        
        test_p = precision_score(y_test, test_preds_binary, zero_division=0)
        test_r = recall_score(y_test, test_preds_binary, zero_division=0)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()
    
    print(f'Epoch {epoch+1:2d} | Loss: {total_loss:.4f} | Val F1: {val_f1:.4f} | Test F1: {test_f1:.4f} | P: {test_p:.4f} R: {test_r:.4f}')

In [ ]:
# Load best model and final evaluation
model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    test_preds = model(torch.tensor(X_test).to(DEVICE)).cpu().numpy()
    test_preds_binary = (test_preds > 0.5).astype(int)
    
    test_f1 = f1_score(y_test, test_preds_binary, zero_division=0)
    test_p = precision_score(y_test, test_preds_binary, zero_division=0)
    test_r = recall_score(y_test, test_preds_binary, zero_division=0)
    test_acc = (test_preds_binary == y_test).mean()

print('\n=== FINAL RESULTS ===')
print(f'Best Val F1: {best_val_f1:.4f}')
print(f'Test F1: {test_f1:.4f}')
print(f'Test Precision: {test_p:.4f}')
print(f'Test Recall: {test_r:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

In [ ]:
# Save model
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_training_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save({
    'model_state_dict': best_model_state,
    'val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': test_p,
    'test_recall': test_r,
}, os.path.join(OUTPUT_DIR, 'wavlm_mlp_model.pt'))

results = {
    'val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': test_p,
    'test_recall': test_r,
    'test_accuracy': test_acc,
    'n_train': len(train_data),
    'n_val': len(val_data),
    'n_test': len(test_data),
    'n_train_videos': len(train_vids),
    'n_val_videos': len(val_vids),
    'n_test_videos': len(test_vids),
}

with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nResults saved to {OUTPUT_DIR}')